In [1]:
import numpy as np
import pandas as pd

# Load the datasets created by data_analysis.ipynb.
processed_data = np.load("data/processed_datasets.npz")
X_train = pd.DataFrame(processed_data["X_train"])
X_val = pd.DataFrame(processed_data["X_val"])
X_test = pd.DataFrame(processed_data["X_test"])
Y_train = pd.Series(processed_data["Y_train"], name="churn")
Y_val = pd.Series(processed_data["Y_val"], name="churn")
Y_test = pd.Series(processed_data["Y_test"], name="churn")
processed_data.close()

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)
print("Y_train:", Y_train.shape)
print("Y_val:", Y_val.shape)
print("Y_test:", Y_test.shape)

X_train: (659028, 44)
X_val: (82379, 44)
X_test: (82379, 44)
Y_train: (659028,)
Y_val: (82379,)
Y_test: (82379,)


In [2]:
print(Y_train.value_counts())

churn
0    593685
1     65343
Name: count, dtype: int64


In [3]:
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

In [4]:
# Keep the balanced variants and the same model settings as the balance comparison.
class_counts = Y_train.value_counts()
positive_class_weight = class_counts[0] / class_counts[1]

models = {
    "logistic_regression": LogisticRegression(
        max_iter=1000, random_state=42, class_weight="balanced"
    ),
    "random_forest": RandomForestClassifier(
        n_estimators=200, max_depth=10, random_state=42, n_jobs=-1,
        class_weight="balanced"
    ),
    "xgboost": XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8,
        random_state=42, eval_metric="logloss", n_jobs=-1,
        scale_pos_weight=positive_class_weight
    ),
}
print("XGBoost positive class weight:", positive_class_weight)

XGBoost positive class weight: 9.085670997658509


In [5]:
N_FEATURES = 25
THRESHOLDS = (0.5, 0.4, 0.3, 0.2, 0.1)
assert X_train.shape[1] >= N_FEATURES

In [6]:
def create_pipeline(model):
    return Pipeline([
        ("feature_selection", SelectKBest(score_func=f_classif, k=N_FEATURES)),
        ("model", model),
    ])

In [7]:
pipelines = {
    model_name: create_pipeline(model)
    for model_name, model in models.items()
}
print("\n".join(f"{model_name}_balanced_top_{N_FEATURES}" for model_name in pipelines))

logistic_regression_balanced_top_25
random_forest_balanced_top_25
xgboost_balanced_top_25


In [8]:
import mlflow
import mlflow.sklearn
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score,
    f1_score, precision_score, recall_score, roc_auc_score,
)

mlflow.set_tracking_uri("sqlite:///mlflow_churn.db")
mlflow.set_experiment("churn-models-balanced-thresholds")


def evaluate_probabilities(y, probabilities, threshold):
    predictions = (probabilities >= threshold).astype(int)
    return {
        "accuracy": accuracy_score(y, predictions),
        "balanced_accuracy": balanced_accuracy_score(y, predictions),
        "precision": precision_score(y, predictions, zero_division=0),
        "recall": recall_score(y, predictions, zero_division=0),
        "f1": f1_score(y, predictions, zero_division=0),
        "roc_auc": roc_auc_score(y, probabilities),
        "average_precision": average_precision_score(y, probabilities),
    }


results = []

/home/rami/miniconda3/envs/mlops/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
for model_name, pipeline in pipelines.items():
    print("Training:", model_name, flush=True)
    pipeline.fit(X_train, Y_train)
    probabilities = pipeline.predict_proba(X_val)[:, 1]

    for threshold in THRESHOLDS:
        run_name = f"{model_name}_balanced_top_{N_FEATURES}_threshold_{threshold}"
        metrics = evaluate_probabilities(Y_val, probabilities, threshold)
        with mlflow.start_run(run_name=run_name):
            mlflow.log_params({
                "model_type": model_name,
                "balance": "balanced",
                "n_features": N_FEATURES,
                "threshold": threshold,
                "evaluation_split": "validation",
            })
            mlflow.log_metrics(metrics)
            # One saved model per fitted pipeline; lower thresholds use its probabilities.
            if threshold == THRESHOLDS[0]:
                mlflow.sklearn.log_model(
                    pipeline, name="model", serialization_format="cloudpickle"
                )

        results.append({
            "model": model_name,
            "balance": "balanced",
            "n_features": N_FEATURES,
            "threshold": threshold,
            **metrics,
        })
        print("Finished:", run_name, "F1:", round(metrics["f1"], 4), flush=True)

Training: logistic_regression


/home/rami/miniconda3/envs/mlops/lib/python3.14/site-packages/sklearn/feature_selection/_univariate_selection.py:110: UserWarning: Features [23] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/home/rami/miniconda3/envs/mlops/lib/python3.14/site-packages/sklearn/feature_selection/_univariate_selection.py:111: RuntimeWarning: invalid value encountered in divide
  f = msb / msw
2026/09/28 18:13:47 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Finished: logistic_regression_balanced_top_25_threshold_0.5 F1: 0.2528
Finished: logistic_regression_balanced_top_25_threshold_0.4 F1: 0.226
Finished: logistic_regression_balanced_top_25_threshold_0.3 F1: 0.202
Finished: logistic_regression_balanced_top_25_threshold_0.2 F1: 0.1843
Finished: logistic_regression_balanced_top_25_threshold_0.1 F1: 0.1804
Training: random_forest


/home/rami/miniconda3/envs/mlops/lib/python3.14/site-packages/sklearn/feature_selection/_univariate_selection.py:110: UserWarning: Features [23] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/home/rami/miniconda3/envs/mlops/lib/python3.14/site-packages/sklearn/feature_selection/_univariate_selection.py:111: RuntimeWarning: invalid value encountered in divide
  f = msb / msw
2026/09/28 18:14:36 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Finished: random_forest_balanced_top_25_threshold_0.5 F1: 0.2511
Finished: random_forest_balanced_top_25_threshold_0.4 F1: 0.2219
Finished: random_forest_balanced_top_25_threshold_0.3 F1: 0.1988
Finished: random_forest_balanced_top_25_threshold_0.2 F1: 0.1806
Finished: random_forest_balanced_top_25_threshold_0.1 F1: 0.1804
Training: xgboost


/home/rami/miniconda3/envs/mlops/lib/python3.14/site-packages/sklearn/feature_selection/_univariate_selection.py:110: UserWarning: Features [23] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/home/rami/miniconda3/envs/mlops/lib/python3.14/site-packages/sklearn/feature_selection/_univariate_selection.py:111: RuntimeWarning: invalid value encountered in divide
  f = msb / msw
2026/09/28 18:14:48 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Finished: xgboost_balanced_top_25_threshold_0.5 F1: 0.2543
Finished: xgboost_balanced_top_25_threshold_0.4 F1: 0.2282
Finished: xgboost_balanced_top_25_threshold_0.3 F1: 0.2046
Finished: xgboost_balanced_top_25_threshold_0.2 F1: 0.1852
Finished: xgboost_balanced_top_25_threshold_0.1 F1: 0.1805


In [10]:
results_df = pd.DataFrame(results).sort_values(
    by=["model", "threshold"], ascending=[True, False]
).reset_index(drop=True)
results_df.to_csv("data/balanced_threshold_results.csv", index=False)
results_df

,model,balance,n_features,threshold,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,average_precision
0,logistic_regression,balanced,25,0.5,0.626482,0.631271,0.157670,0.637243,0.252793,0.682128,0.201936
1,logistic_regression,balanced,25,0.4,0.442091,0.610915,0.131024,0.821499,0.226002,0.682128,0.201936
2,logistic_regression,balanced,25,0.3,0.268527,0.564482,0.113243,0.933643,0.201986,0.682128,0.201936
3,logistic_regression,balanced,25,0.2,0.128844,0.513211,0.101587,0.992654,0.184313,0.682128,0.201936
4,logistic_regression,balanced,25,0.1,0.099151,0.500000,0.099151,1.000000,0.180415,0.682128,0.201936
5,random_forest,balanced,25,0.5,0.613834,0.631278,0.155456,0.653036,0.251130,0.680508,0.201240
6,random_forest,balanced,25,0.4,0.417060,0.604594,0.127894,0.838516,0.221938,0.680508,0.201240
7,random_forest,balanced,25,0.3,0.250003,0.556435,0.111197,0.938663,0.198838,0.680508,0.201240
8,random_forest,balanced,25,0.2,0.100220,0.500484,0.099238,0.999755,0.180554,0.680508,0.201240
9,random_forest,balanced,25,0.1,0.099151,0.500000,0.099151,1.000000,0.180415,0.680508,0.201240
